Подготовка датасета для задачи QA

In [ ]:
import os
import json
import random
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import re

In [ ]:
def prepare_stackexchange_data(input_file="stackexchange.jsonl",
                               output_train="stackexchange_train.jsonl",
                               output_test="stackexchange_test.jsonl",
                               test_size=0.1,
                               max_samples=None):

    data = []

    with open(input_file, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f):
            if max_samples and len(data) >= max_samples:
                break

            try:
                item = json.loads(line.strip())

                #определение структуры
                if isinstance(item, list) and len(item) >= 2:
                    question = item[0].strip()
                    answer = item[1].strip()
                elif isinstance(item, dict):
                    question = item.get('question', item.get('title', item.get('input', ''))).strip()
                    answer = item.get('answer', item.get('best_answer', item.get('output', ''))).strip()
                else:
                    print(f"Строка {line_num}: неизвестный формат, пропускаем")
                    continue

                #фильтр пустых и слишком коротких
                if not question or not answer:
                    continue

                #формирование input как в датасете
                input_text = f"Title: {question[:100]}\n\nQuestion:\n{question}"

                data.append({
                    "input": input_text,
                    "output": answer
                })

            except json.JSONDecodeError as e:
                print(f"Ошибка JSON в строке {line_num}: {e}")
                continue

    #разделяем на выборки
    train_data, test_data = train_test_split(data, test_size=test_size, random_state=42)

    #сохраянем результат
    def save_jsonl(data_list, filepath):
        with open(filepath, "w", encoding="utf-8") as f:
            for example in data_list:
                f.write(json.dumps(example, ensure_ascii=False) + "\n")

    save_jsonl(train_data, output_train)
    save_jsonl(test_data, output_test)

    return train_data, test_data

In [ ]:
train_data, test_data = prepare_stackexchange_data(
    input_file="stackexchange.jsonl",
    output_train="stackexchange_train.jsonl",
    output_test="stackexchange_test.jsonl",
)